# Level 2: Data Cleaning and Feature Engineering

The source file contains one row per train stop. This notebook validates the source data, reconstructs one row per train journey, creates duration features, and exports the train-level dataset used by the later analysis.

In [ ]:
import numpy as np
import pandas as pd

source_path = '../data/Internship Dataset.csv'
output_path = '../data/train_level2_cleaned.csv'
df = pd.read_csv(source_path)
df.head()

## Source-data quality checks

In [ ]:
print(f'Records: {len(df):,}')
print(f'Columns: {df.shape[1]}')
print(f'Unique trains: {df.Train_No.nunique():,}')
print(f'Duplicate rows: {df.duplicated().sum():,}')
display(df.isna().sum().to_frame('missing_values'))
display(df.describe(include='all').T)

In [ ]:
df = df.sort_values(['Train_No', 'SN']).copy()
sn_order = df.groupby('Train_No')['SN'].apply(lambda values: values.is_monotonic_increasing)
route_counts = df.groupby('Train_No')['Route_Number'].nunique()
print(f'Trains with ordered station records: {sn_order.sum():,} / {len(sn_order):,}')
print(f'Trains with multiple route numbers: {(route_counts > 1).sum():,}')
print(f'Negative distances: {(df.Distance < 0).sum():,}')

## Reconstruct train-level journeys

In [ ]:
journeys = df.groupby('Train_No', sort=False).agg(
    Start_Station=('Station_Name', 'first'),
    End_Station=('Station_Name', 'last'),
    Start_Departure=('Departure_Time', 'first'),
    End_Arrival=('Arrival_time', 'last'),
    Total_Distance=('Distance', 'max'),
    Number_of_Stops=('Station_Name', 'size'),
).reset_index()
journeys.head()

In [ ]:
start_time = pd.to_timedelta(journeys['Start_Departure'])
end_time = pd.to_timedelta(journeys['End_Arrival'])
duration = end_time - start_time
overnight = duration < pd.Timedelta(0)
journeys['Journey_Duration'] = duration.where(~overnight, duration + pd.Timedelta(days=1))
journeys['Journey_Hours'] = journeys['Journey_Duration'].dt.total_seconds() / 3600
print(f'Overnight journeys adjusted: {overnight.sum():,}')
journeys[['Journey_Duration', 'Journey_Hours']].describe()

In [ ]:
feature_columns = [
    'Train_No', 'Start_Station', 'End_Station', 'Start_Departure',
    'End_Arrival', 'Total_Distance', 'Number_of_Stops', 'Journey_Duration'
]
train_level = journeys[feature_columns].copy()
train_level.to_csv(output_path, index=False)
print(f'Saved {len(train_level):,} journeys to {output_path}')
print(f'Missing values: {train_level.isna().sum().sum():,}')
print(f'Duplicate train numbers: {train_level.Train_No.duplicated().sum():,}')